<a href="https://colab.research.google.com/github/hitarthi45/GENAI/blob/main/Shoolini_Pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets transformers tokenizer accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 8.6 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
import os

In [ ]:
data=load_dataset('text',data_files='/content/adm_data.csv')

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
data

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 401
    })
})

In [ ]:
from transformers import (
    GPT2TokenizerFast,
    GPT2Config,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

In [ ]:
tokenizer=GPT2TokenizerFast.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
tokenizer.pad_token=tokenizer.eos_token

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"])

In [ ]:
tokenized_data=data.map(tokenize_function,batched=True,num_proc=4,remove_columns=["text"])

Map (num_proc=4):   0%|          | 0/401 [00:00<?, ? examples/s]

In [ ]:
tokenized_data

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 401
    })
})

In [ ]:
config=GPT2Config(
    vocab_size=tokenizer.vocab_size,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    n_layer=6,
    n_head=6,
    n_embd=384
)

new_model=GPT2LMHeadModel(config)


In [ ]:
data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)

In [ ]:
training_args=TrainingArguments(
    output_dir="./gpt2-shoolini",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    eval_steps=50,
    save_total_limit=2
)

In [ ]:
trainer=Trainer(
    model=new_model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_data["train"]
)

In [ ]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=65, training_loss=6.806388502854567, metrics={'train_runtime': 7.5093, 'train_samples_per_second': 267.004, 'train_steps_per_second': 8.656, 'total_flos': 3253359513600.0, 'train_loss': 6.806388502854567, 'epoch': 5.0})

In [ ]:
new_model.save_pretrained("./gpt2-adm_data")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
prompt="What is the process of admission in university"

In [ ]:
tokenizer.save_pretrained("./gpt2-adm_data")

('./gpt2-adm_data/tokenizer_config.json', './gpt2-adm_data/tokenizer.json')

In [ ]:
from transformers import pipeline


text_generator = pipeline("text-generation", model="./gpt2-adm_data")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [ ]:
output=text_generator(prompt, max_length=50, do_sample=True)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print(output[0]['generated_text'])

What is the process of admission in university,,,, Wel,3,4,0,0...radius, smuggling......,0.,0..,0...5,0.......301,0..0..,0.00. Rein,0..0.. Franc diabetes,0.0...,0NOTE,0.....borgh clam,0Merc Pyro0 Knicks Startup.0..00.0:',0. includeenza,0OUT.. neglig...0.ometers0.. Alexander,0.ウス,Faith impossibility.. Rhod. RhodLib,0 platform,00 Tw0 Knicks IncomeJs,0herical. Elm,0.. pregnanciesJs T,1.... OTHER,EMBER... platform,0 timing OTHER0.322 rec0.. Alexander.herical,00.00.0 platform.46Js,0enza Chem complying,�..... neglig. Rein,0..0.365enzabiology Blockchain.. OTHER... Rhod,0 reviews0.e,0 clam clam redevelop complying,00biology,quest..114.
